In [ ]:
# import libraries
import geopandas as gpd
from pathlib import Path
from pystac import Collection
import pystac
import xarray as xr
from datetime import datetime
from glob import glob
from shapely.geometry import box, mapping
import os
from datetime import timezone

## Create the EarthCARE-frame lightning collection 
to which the files/STAC items will be added.

In [ ]:
collectionid = "earthcare-frame-lightning"
description = "The EarthCARE-frame lightning product provides geostationary satellite-detected lightning groups collocated with individual EarthCARE MSI-like observation frames. It includes all lightning activity occurring within the frame and a surrounding 0.5° bounding box within ±1 hour of the EarthCARE overpass. The product is distributed as monthly Parquet files, with MTG-LI and GOES-GLM observations stored separately. The dataset contains lightning groups together with their time, location, radiance, and associated flash identifier. Additional variables describe the temporal difference from the EarthCARE overpass, parallax-corrected lightning positions for groups close to the overpass, and clustering information used to identify lightning structures within storms."
collection = Collection.from_dict(
    
{
  "type": "Collection",
  "id": collectionid,
  "stac_version": "1.1.0",
  "title": "EarthCARE-frame lightning",
  "description": description,
  "extent": {
    "spatial": {
      "bbox": [
        [-180, 
         -60, 
         180,
         60]
      ]
    },
    "temporal": {
      "interval": [
        [
          "2024-08-01T00:00:00Z",
          "2026-02-01T00:00:00Z"
        ]
      ]
    }
  },
  "license": "CC-BY-4.0",
  "links": [],
  "created": "2026-03-17T00:00:00Z",
  "summaries": {
    "platform": [
      "EarthCARE",
      "MTG-I1",
      "GOES-16",
      "GOES-18",
      "GOES-19",
    ],
    "instruments": [
      "MSI", 
      "LI", 
      "GLM"
    ]
  }

}

)

collection

In [ ]:
collection.validate()

Extract all variable metadata from sample nc files.

In [ ]:
lightning_path = Path("/path/to/lightning_groups")
glm_file = next(lightning_path.glob("GLM_*.nc"))
li_file = next(lightning_path.glob("LI_*.nc"))

glm = xr.open_dataset(glm_file)
glm_vars = {}
for var in glm.variables:
    glm_vars[var] = glm[var].attrs
li = xr.open_dataset(li_file)
li_vars = {}
for var in li.variables:
    li_vars[var] = li[var].attrs

Generate stac items per file type and add to the collection.

In [ ]:
prefix = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/storm-data/"
parquet_path = Path("path/to/parquet")

c1_li_files = sorted(parquet_path.glob("EC_lightning_LI*.parquet"))
c1_glm_files = sorted(parquet_path.glob("EC_lightning_GLM*.parquet"))

In [ ]:
# crete STAC items for the C1 LI files collection

for file in c1_li_files:

    base_name = os.path.basename(file)
    item_id = os.path.splitext(base_name)[0]
    gdf = gpd.read_parquet(file)
    bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
    geom = mapping(box(*bounds))

    parts = item_id.split("_")
    year = int(parts[-2])
    month = int(parts[-1])

    start_dt = datetime(year, month, 1, tzinfo=timezone.utc)

    if month == 12:
        end_dt = datetime(year + 1, 1, 1, tzinfo=timezone.utc)
    else:
        end_dt = datetime(year, month + 1, 1, tzinfo=timezone.utc)

    properties = {
        "parquet:columns": li_vars,
        "source": "MTG-LI",
        "type": "Lightning groups within EarthCARE MSI frame",
        "start_datetime": start_dt.isoformat().replace("+00:00", "Z"),
        "end_datetime": end_dt.isoformat().replace("+00:00", "Z"),
        "created": "2026-03-17T00:00:00Z"
    }

    # Create the STAC Item
    item = pystac.Item(
        id=item_id,
        geometry=geom,
        bbox=bounds,
        datetime=start_dt,
        properties=properties
    )

    # add the asset
    path = prefix + file.split("/")[-1]
    asset_href = path
    asset = pystac.Asset(
        href=asset_href,
        media_type="application/x-parquet",
        roles=["data"]
    )

    item.add_asset(key="data", asset=asset) 
    collection.add_item(item)
    print(f"  - Added Item: {item_id}")


In [ ]:
## crete STAC items for the C1 GLM files collection

bounds = [-180, -60, 180, 60]
geom = {
    "type": "MultiPolygon",
    "coordinates": [
        [  # western side (170 → 180)
            [
                [165, -60],
                [180, -60],
                [180, 60],
                [165, 60],
                [165, -60]
            ]
        ],
        [  # eastern side (-180 → -20)
            [
                [-180, -60],
                [-20, -60],
                [-20, 60],
                [-180, 60],
                [-180, -60]
            ]
        ]
    ]
}


for file in c1_glm_files:

    base_name = os.path.basename(file)
    item_id = os.path.splitext(base_name)[0] 

    parts = item_id.split("_")
    year = int(parts[-2])
    month = int(parts[-1])

    start_dt = datetime(year, month, 1, tzinfo=timezone.utc)

    if month == 12:
        end_dt = datetime(year + 1, 1, 1, tzinfo=timezone.utc)
    else:
        end_dt = datetime(year, month + 1, 1, tzinfo=timezone.utc)

    properties = {
        "parquet:columns": glm_vars,
        "source": "GOES-GLM",
        "type": "Lightning groups within EarthCARE MSI frame",
        "start_datetime": start_dt.isoformat().replace("+00:00", "Z"),
        "end_datetime": end_dt.isoformat().replace("+00:00", "Z"),
        "created": "2026-03-17T00:00:00Z"
    }

    item = pystac.Item(
        id=item_id,
        geometry=geom,
        bbox=bounds,
        datetime=start_dt,
        properties=properties
    )

    path = prefix + file.split("/")[-1]
    asset_href = path
    asset = pystac.Asset(
        href=asset_href,
        media_type="application/x-parquet",
        roles=["data"]
    )
    
    item.add_asset(key="data", asset=asset) 
    collection.add_item(item)
    print(f"  - Added Item: {item_id}")


In [ ]:
# save the collection
collection.normalize_and_save(
    root_href='./earthcare-frame-lightning/',
    catalog_type=pystac.CatalogType.SELF_CONTAINED
)

## Create the EarthCARE along-track lightning counts collection 

In [ ]:
collectionid = "earthcare-along-track-lightning"
description = "The along-track lightning counts product provides lightning statistics collocated with EarthCARE CPR observations along the satellite nadir track. For each CPR sample, the dataset quantifies the number of lightning groups detected by geostationary satellites within defined spatial and temporal windows centred on the observation point. The dataset is distributed as Parquet files, with separate files for each lightning data source (MTG-LI and GOES-GLM)."
collection = Collection.from_dict(
    
{
  "type": "Collection",
  "id": collectionid,
  "stac_version": "1.1.0",
  "title": "EarthCARE along-track lightning counts",
  "description": description,
  "extent": {
    "spatial": {
      "bbox": [
        [-180, 
         -60, 
         180,
         60]
      ]
    },
    "temporal": {
      "interval": [
        [
          "2024-08-01T00:00:00Z",
          "2026-02-01T00:00:00Z"
        ]
      ]
    }
  },
  "license": "CC-BY-4.0",
  "links": [],
  "created": "2026-03-17T00:00:00Z",
  "summaries": {
    "platform": [
      "EarthCARE",
      "MTG-I1",
      "GOES-16",
      "GOES-18",
      "GOES-19",
    ],
    "instruments": [
      "CPR",
      "ATLID",
      "LI", 
      "GLM"
    ]
  }

}

)
collection

Extract all variable metadata from sample nc files.

In [ ]:
track_path = Path("/path/to/track_counts")
glm_track = next(track_path.glob("CPR-GLM-sum_*.nc"))
li_track = next(track_path.glob("CPR-LI-sum_*.nc"))

c2li = xr.open_dataset(li_track)
c2li_vars = {}
for var in c2li.variables:
    c2li_vars[var] = c2li[var].attrs
c2glm = xr.open_dataset(glm_track)
c2glm_vars = {}
for var in c2glm.variables:
    c2glm_vars[var] = c2glm[var].attrs

Generate stac items per file type and add to the collection.

In [ ]:
c2_glm_file = parquet_path / "EC_track_lightning_GLM.parquet"
c2_li_file = parquet_path / "EC_track_lightning_LI.parquet"

In [ ]:
file = c2_glm_file
base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 

bounds = [-180, -60, 180, 60]
geom = {
    "type": "MultiPolygon",
    "coordinates": [
        [  # western side (170 → 180)
            [
                [165, -60],
                [180, -60],
                [180, 60],
                [165, 60],
                [165, -60]
            ]
        ],
        [  # eastern side (-180 → -20)
            [
                [-180, -60],
                [-20, -60],
                [-20, 60],
                [-180, 60],
                [-180, -60]
            ]
        ]
    ]
}

properties = {
    "parquet:columns": c2glm_vars,
    "source": "GOES-GLM",
    "type": "Lightning groups count along EarthCARE nadir track",
    "start_datetime": "2024-08-01T00:00:00Z",
    "end_datetime": "2026-02-01T00:00:00Z",
    "created": "2026-03-17T00:00:00Z"
}

item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=datetime(2024, 8, 1, 0, 0, 0, tzinfo=timezone.utc),
    properties=properties
)
path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset)
collection.add_item(item)
print(f"  - Added Item: {item_id}")

In [ ]:
file = c2_li_file

base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 
gdf = gpd.read_parquet(file)
bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]

geom = mapping(box(*bounds))

properties = {
    "parquet:columns": c2li_vars,
    "source": "MTG-LI",
    "type": "Lightning groups count along EarthCARE nadir track",
    "start_datetime": "2024-08-01T00:00:00Z",
    "end_datetime": "2026-02-01T00:00:00Z",
    "created": "2026-03-17T00:00:00Z"
}

item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=datetime(2024, 8, 1, 0, 0, 0, tzinfo=timezone.utc),
    properties=properties
)

path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset) 
collection.add_item(item)
print(f"  - Added Item: {item_id}")

In [ ]:
# save the collection
collection.normalize_and_save(
    root_href='./earthcare-along-track-lightning/',
    catalog_type=pystac.CatalogType.SELF_CONTAINED
)

## Create the EarthCARE lightning storm catalogue collection 

In [ ]:
collectionid = "earthcare-storm-catalogue"
description = "The lightning storm catalogue provides a set of lightning clusters sampled along the EarthCARE nadir track by its active sensors (CPR and ATLID). Each cluster represents an electrically active region within a convective system, such as an individual convective cell, anvil lightning, or a broader multicellular storm area. The dataset is distributed as Parquet files: one containing cluster-level descriptors (e.g., lightning counts, spatial extent) and another describing the temporal evolution of lightning activity within each cluster in a ±1 hour window around the EarthCARE overpass."
collection = Collection.from_dict(
    
{
  "type": "Collection",
  "id": collectionid,
  "stac_version": "1.1.0",
  "title": "EarthCARE lightning storm catalogue",
  "description": description,
  "extent": {
    "spatial": {
      "bbox": [
        [-180, 
         -60, 
         180,
         60]
      ]
    },
    "temporal": {
      "interval": [
        [
          "2024-08-01T00:00:00Z",
          "2026-02-01T00:00:00Z"
        ]
      ]
    }
  },
  "license": "CC-BY-4.0",
  "links": [],
  "created": "2026-03-17T00:00:00Z",
  "summaries": {
    "platform": [
      "EarthCARE",
      "MTG-I1",
      "GOES-16",
      "GOES-18",
      "GOES-19",
    ],
    "instruments": [
      "MSI",
      "CPR",
      "ATLID",
      "LI", 
      "GLM"
    ]
  }

}

)
collection

All variable metadata

In [ ]:
c3 = {
    "unique_id": "Unique identifier of the lightning cluster (earthcare_id + source + cluster_id)",  
    "earthcare_id": "Orbit/frame identifier of the EarthCARE overpass",
    "source": "Lightning data source: LI (MTG) or GLM (GOES)",
    "parent_cluster_id": "Parent lightning cluster identifier",
    "cluster_id": "Lightning cluster identifier",
    "surface_type": "Surface classification at cluster location (land, water, coast)",
    "peak_datetime": "Time of CPR sample with maximum lightning activity",
    "peak_lat": "Latitude of CPR sample with maximum lightning activity",
    "peak_lon": "Longitude of CPR sample with maximum lightning activity",
    "peak_lightning": "Maximum lightning group count at a CPR sample",
    "nadir_lightning": "Lightning group count within ±2.5 min and ±2.5 km of CPR nadir track",
    "cluster_lightning": "Lightning group count within ±2.5 min of CPR peak time (any distance)",
    "cluster_area_km2": "Area of cluster in km2",
    "cluster_mean_lat": "Mean latitude of lightning groups within ±2.5 min of CPR peak time",
    "cluster_mean_lon": "Mean longitude of lightning groups within ±2.5 min of CPR peak time",
    "cluster_dist_km": "Minimum distance (km) between CPR track and mean cluster location",
    "first_lightning_min": "Minute offset of first lightning in parent cluster relative to peak_datetime",
    "last_lightning_min": "Minute offset of last lightning in parent cluster relative to peak_datetime",
    "duration_min": "Parent cluster duration in minutes",
    "travel_km": "Shortest distance (km) between first and last lightning locations in parent cluster",
    "minute_counts": "Lightning group counts in parent cluster per minute (−60…+60) relative to peak_datetime",
    "minute_mean_lat": "Mean lightning latitude in parent cluster per minute (−60…+60) relative to peak_datetime",
    "minute_mean_lon": "Mean lightning longitude in parent cluster per minute (−60…+60) relative to peak_datetime",
    "missing_peak_minutes": "Total minutes of missing data overlapping ±2.5 min around peak_datetime",
    "masked_minute_counts": "Lightning group counts per minute (−60…+60) with missing intervals masked as null",
    "minute_offset": "Minute offset (−60…+60) relative to peak_datetime",
    "absolute_time": "Absolute time corresponding to minute_offset relative to peak_datetime",
    "lightning_count": "Lightning group count in parent cluster per minute (−60…+60) relative to peak_datetime",
    "masked_lightning_count": "Lightning group count in parent cluster per minute (−60…+60) relative to peak_datetime, with missing intervals masked as null",
    "latitude": "Mean lightning latitude in parent cluster per minute (−60…+60) relative to peak_datetime",
    "longitude": "Mean lightning longitude in parent cluster per minute (−60…+60) relative to peak_datetime",
  }

Generate stac items per file type and add to the collection.

In [ ]:
c3evol_file = parquet_path / "EC_lightning_cluster_evolution.parquet"
c3summary_file = parquet_path / "EC_lightning_clusters.parquet"

In [ ]:
c3evol = gpd.read_parquet(c3evol_file)
c3summary = gpd.read_parquet(c3summary_file)
list(c3evol.columns)
list(c3summary.columns)
c3evol_descriptions = {}
for col in c3evol.columns:
    if col in c3:
        c3evol_descriptions[col] = c3[col]
    
c3summary_descriptions = {}
for col in c3summary.columns:
    if col in c3:
        c3summary_descriptions[col] = c3[col]

Add the summary and the evolution files

In [ ]:
file = c3summary_file

base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 
bounds = [-180, -60, 180, 60]
geom = {
    "type": "MultiPolygon",
    "coordinates": [
        [  # western side (170 → 180)
            [
                [165, -60],
                [180, -60],
                [180, 60],
                [165, 60],
                [165, -60]
            ]
        ],
        [  # eastern side (-180 → 60)
            [
                [-180, -60],
                [60, -60],
                [60, 60],
                [-180, 60],
                [-180, -60]
            ]
        ]
    ]
}

properties = {
    "parquet:columns": c3summary_descriptions,
    "type": "Lightning storm catalogue statistics",
    "start_datetime": "2024-08-01T00:00:00Z",
    "end_datetime": "2026-02-01T00:00:00Z",
    "created": "2026-03-17T00:00:00Z"
}
item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=datetime(2024, 8, 1, 0, 0, 0, tzinfo=timezone.utc),
    properties=properties
)

path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset) 
collection.add_item(item)
print(f"  - Added Item: {item_id}")

In [ ]:
file = c3evol_file

base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 
bounds = [-180, -60, 180, 60]
geom = {
    "type": "MultiPolygon",
    "coordinates": [
        [  # western side (170 → 180)
            [
                [165, -60],
                [180, -60],
                [180, 60],
                [165, 60],
                [165, -60]
            ]
        ],
        [  # eastern side (-180 → 60)
            [
                [-180, -60],
                [60, -60],
                [60, 60],
                [-180, 60],
                [-180, -60]
            ]
        ]
    ]
}

properties = {
    "parquet:columns": c3evol_descriptions,
    "type": "Lightning storm catalogue evolution statistics",
    "start_datetime": "2024-08-01T00:00:00Z",
    "end_datetime": "2026-02-01T00:00:00Z",
    "created": "2026-03-17T00:00:00Z"
}
item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=datetime(2024, 8, 1, 0, 0, 0, tzinfo=timezone.utc),
    properties=properties
)

path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset) 
collection.add_item(item)
print(f"  - Added Item: {item_id}")

In [ ]:
# save the collection
collection.normalize_and_save(
    root_href='./earthcare-storm-catalogue/',
    catalog_type=pystac.CatalogType.SELF_CONTAINED
)